In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from torch.utils.data import TensorDataset, DataLoader

In [2]:
df = pd.read_csv('/home/tori/behavioural-fruad-detection/data/raw/train.csv')
df2 = pd.read_csv('/home/tori/behavioural-fruad-detection/data/raw/test.csv')

df['is_test'] = 0
df2['is_test'] = 1

full = pd.concat([df, df2])
full = full.sort_values(by=["user_id", "timestamp"]).reset_index(drop=True)
full.shape

(227656, 11)

In [3]:
full['is_first_txn'] = (full['hours_since_prev_txn'] == 999).astype(int)
full['hours_since_prev_txn'] = full['hours_since_prev_txn'].replace(999, np.nan)

#spending spike
full['running_total'] = full.groupby('user_id')['amount'].cumsum()
full['past_total'] = full['running_total'] - full['amount']
full['past_count'] = full.groupby('user_id').cumcount()
full['past_avg_amount'] = full['past_total'] / full['past_count']
full['amount_ratio'] = full['amount'] / full['past_avg_amount']

#new device/country/merchant
full['is_new_device'] = (full.groupby(['user_id', 'device_id']).cumcount() == 0).astype(int)
full['is_new_country'] = (full.groupby(['user_id', 'country']).cumcount() == 0).astype(int)
full['is_new_merchant'] = (full.groupby(['user_id', 'merchant_category']).cumcount() == 0).astype(int)

full['prev_gap_1'] = full.groupby('user_id')['hours_since_prev_txn'].shift(1)
full['prev_gap_2'] = full.groupby('user_id')['hours_since_prev_txn'].shift(2)
full['avg_gap_last3'] = full[['hours_since_prev_txn', 'prev_gap_1', 'prev_gap_2']].mean(axis=1)

#clean
full = full.drop(columns=['running_total', 'past_total', 'past_count', 'prev_gap_1', 'prev_gap_2'])

full.shape

(227656, 18)

In [4]:
full['hours_since_prev_txn'] = full['hours_since_prev_txn'].fillna(0)
full['amount_ratio'] = full['amount_ratio'].fillna(1)

full['amount_log'] = np.log1p(full['amount'])
full['gap_log'] = np.log1p(full['hours_since_prev_txn'])
full['ratio_log'] = np.log1p(full['amount_ratio'])

full['amount_log'] = (full['amount_log'] - full['amount_log'].mean()) / full['amount_log'].std()
full['gap_log'] = (full['gap_log'] - full['gap_log'].mean()) / full['gap_log'].std()
full['ratio_log'] = (full['ratio_log'] - full['ratio_log'].mean()) / full['ratio_log'].std()

full['present'] = 1
seq_cols = ['amount_log', 'gap_log', 'ratio_log', 'channel', 'is_first_txn',
            'is_new_device', 'is_new_country', 'is_new_merchant', 'present']

In [5]:
steps = []
for k in range(19, -1, -1):
    shifted = full.groupby('user_id')[seq_cols].shift(k).fillna(0).to_numpy()
    steps.append(shifted)
    
x_seq = np.stack(steps, axis=1)

In [6]:
labeled_idx = np.where(full['is_test'] == 0)[0]
y_labeled = full[full['is_test'] == 0]['label'].to_numpy()

train_idx, val_idx = train_test_split(labeled_idx, test_size=0.2, stratify=y_labeled, random_state=42)

x_train_seq = x_seq[train_idx]
x_val_seq = x_seq[val_idx]

all_labels = full['label'].to_numpy()
y_train_seq = all_labels[train_idx]
y_val_seq = all_labels[val_idx]

In [7]:
xtr = torch.tensor(x_train_seq, dtype=torch.float32)
xva = torch.tensor(x_val_seq, dtype=torch.float32)
ytr = torch.tensor(y_train_seq, dtype=torch.float32)
yva = torch.tensor(y_val_seq, dtype=torch.float32)

train_ds = TensorDataset(xtr, ytr)
val_ds = TensorDataset(xva, yva)

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False)

In [8]:
xb, yb = next(iter(train_loader))
xb.shape

torch.Size([512, 20, 9])

In [9]:
class FraudLSTM(nn.Module):
    def __init__(self, n_features=9, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden, batch_first=True)
        self.out = nn.Linear(hidden, 1)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.out(h[-1]).squeeze(1)

In [10]:
lstm_model = FraudLSTM()

In [11]:
lstm_model(xb).shape

torch.Size([512])

In [16]:
torch.manual_seed(3)
lstm_model = FraudLSTM()

pos_weight = torch.tensor([(ytr == 0).sum() / (ytr == 1).sum()])
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)

def evaluate(model, loader):
    model.eval()
    probs = []
    with torch.no_grad():
        for xb, yb in loader:
            probs.append(torch.sigmoid(model(xb)))
    return torch.cat(probs).numpy()

for epoch in range(5):
    lstm_model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = lstm_model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()

    val_probs = evaluate(lstm_model, val_loader)
    print(epoch, round(loss.item(), 4), average_precision_score(y_val_seq, val_probs))

0 1.0985 0.022357244470606368
1 1.3482 0.02322662820267344
2 1.2307 0.023523751451171006
3 1.7632 0.023958559186615217
4 1.5761 0.024355923788617166
